# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

Valentin Torres

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [3]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    import heapq
    from itertools import count

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    def heuristic(state: StatesHanoi) -> int:
        """Heurística admisible: si k discos grandes ya están correctamente
        asentados al fondo del peg objetivo, faltan al menos 2^(n-k)-1 movimientos."""
        goal_rod = state.rods[2]
        expected = list_disks  # [n, n-1, ..., 1]
        k = 0
        for i in range(min(len(goal_rod), len(expected))):
            if goal_rod[i] == expected[i]:
                k += 1
            else:
                break
        remaining = number_disks - k
        return (2 ** remaining) - 1

    start = NodeHanoi(problem.initial)
    if problem.goal_test(start.state):
        metrics = {
            "solution_found": True,
            "nodes_explored": 0,
            "states_visited": 1,
            "nodes_in_frontier": 0,
            "max_depth": 0,
            "cost_total": float(start.path_cost),
        }
        return start, metrics

    # A* graph search
    tie = count()
    frontier = []  # (f, tie, node)
    heapq.heappush(frontier, (start.path_cost + heuristic(start.state), next(tie), start))

    best_g = {start.state: float(start.path_cost)}

    nodes_explored = 0
    states_visited = 1
    max_depth = 0

    while frontier:
        _, _, node = heapq.heappop(frontier)

        g = float(node.path_cost)
        if g != best_g.get(node.state, float("inf")):
            continue

        nodes_explored += 1
        if node.depth > max_depth:
            max_depth = node.depth

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": states_visited,
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": float(node.path_cost),
            }
            return node, metrics

        for action in problem.actions(node.state):
            child = node.child_node(problem, action)
            child_g = float(child.path_cost)

            prev = best_g.get(child.state)
            if prev is None or child_g < prev:
                if prev is None:
                    states_visited += 1
                best_g[child.state] = child_g
                f = child_g + heuristic(child.state)
                heapq.heappush(frontier, (f, next(tie), child))

    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": states_visited,
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics

Se prueba la función:

In [4]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [5]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 116
states_visited: 125
nodes_in_frontier: 9
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [6]:
if solution is None:
    print("No se encontró solución")
else:
    for nodos in solution.path():
        print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [7]:
if solution is None:
    print("No se encontró solución")
else:
    for act in solution.solution():
        print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
